In [78]:
import numpy as np
import time
import torch
import torch.nn.functional as F
import os
from datasets import load_dataset
from functools import lru_cache
from transformers import AutoModel, AutoTokenizer
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import tenseal as ts
import tenseal.sealapi as seal

MODEL_ID = "Snowflake/snowflake-arctic-embed-m-v1.5"
FULL_DIM = 768 # native dimensionality of the model
NUM_DOCS = 1000
NUM_QUERIES = 200
TOP_K = 10
TIMING_STAGES = ["Encode question:", "Encrypt query:", "Server similarity:",
          "Decrypt scores:", "Top-k selection:", "Fetch and/or decrypt docs:"]
NUM_QUERY_SUBSET = NUM_QUERIES # Can be used to reduce number of queries on the encrypted operations, useful for debugging

device = torch.device("cpu")

In [47]:
def load_corpus_and_queries():
    """Build the doc DB (unique contexts) and query set (questions)."""
    ds = load_dataset("rajpurkar/squad", split="validation")

    # Unique contexts, preserving first-seen order
    contexts, seen = [], set()
    for ex in ds:
        c = ex["context"]
        if c not in seen:
            seen.add(c)
            contexts.append(c)
        if len(contexts) == NUM_DOCS:
            break
    assert len(contexts) == NUM_DOCS

    # Queries: questions whose gold context is in the DB
    ctx_to_id = {c: i for i, c in enumerate(contexts)}
    queries, gold = [], []
    for ex in ds:
        c = ex["context"]
        if c in ctx_to_id:
            queries.append(ex["question"])
            gold.append(ctx_to_id[c])
        if len(queries) == NUM_QUERIES:
            break
    assert len(queries) == NUM_QUERIES
    return contexts, queries, gold

@lru_cache(maxsize=1)
def load_model():
    """
    Loads the tokenizer and model, and caches them so multiple encode() calls don't load more than once.
    """
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
    return tok, model


def encode(texts, is_query=False, batch_size=64):
    """Return one embedding per text, L2-normalized, on CPU.

    Arctic models are *asymmetric*: queries get a prompt prefix,
    documents are encoded as-is (this matches the model card).
    """
    tok, model = load_model()

    if is_query:
        texts = ["Represent this sentence for searching relevant passages: " + t
                 for t in texts]

    vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tok(
                texts[i:i + batch_size],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)
            out = model(**batch)

            # mean pooling over non-padding tokens
            mask = batch["attention_mask"].unsqueeze(-1).float()
            emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
            emb = F.normalize(emb, p=2, dim=1)
            vecs.append(emb)
    return torch.cat(vecs)  # (n, 768), float32, unit norm


def truncate(matryoshka_vecs, dim):
    """Keep the first `dim` components and re-normalize.

    Re-normalization is essential: the prefix has norm < 1 in general,
    and cosine similarity is scale-invariant, so this preserves the
    ranking geometry the model was trained to have.
    """
    return F.normalize(matryoshka_vecs[:, :dim], p=2, dim=1)

# --------------------------------
# Part (c): 1. cosine similarity function
#           2. top_k selection
# --------------------------------

def cosine_similarity(query_vec, doc_matrix):
    """Cosine similarity of one query against all documents.

    Derived from the Euclidian dot product formula:
    A @ B = norm(A) * norm(B) * cos(theta)
    <=>
    cos(theta) = (A @ B) / (norm(A) * norm(B))
    """

    q = np.asarray(query_vec, dtype=np.float64)
    D = np.asarray(doc_matrix, dtype=np.float64)
    dots = D @ q
    qn = np.linalg.norm(q)
    dn = np.linalg.norm(D, axis=1)
    return dots / np.maximum(qn * dn, 1e-12)

def top_k_indices(scores, k):
    """Indices of k largest scores, in descending order"""
    s = np.array(scores, dtype=np.float64, copy=True)
    k = min(k, s.shape[0])
    out = []
    for _ in range(k):
        i = int(np.argmax(s))
        out.append(i)
        s[i] = -np.inf
    return out

def recall_at_k(docs, queries, gold, k=TOP_K):
    """Recall@k: fraction of queries whose gold doc is in the top-k."""
    docs = np.asarray(docs)
    queries = np.asarray(queries)
    hits = 0
    for qi in range(queries.shape[0]):
        sims = cosine_similarity(queries[qi], docs)
        if gold[qi] in top_k_indices(sims, k):
            hits += 1
    return hits / queries.shape[0]

In [48]:
class RetrieverServer:
    """Class that stores the 1,000 documents and their vectors. The client
    may only talk to it through a single search(query_vector, k) call.
    """

    def __init__(self, contexts, embed_dim=256):
        # Embedding the corpus happens here, once.
        self.contexts = list(contexts)
        self.embed_dim = embed_dim
        D_full = encode(self.contexts, is_query=False)     # (1000, 768)

        # Matryoshka truncation to the working dimension, renormalized.
        self.doc_vecs = truncate(D_full, embed_dim)         # (1000, embed_dim)

    def search(self, query_vector, k):
        """Return the ids (and texts) of the k nearest documents.

        query_vector: (embed_dim,) numpy array or float tensor,
        assumed L2-normalized (the client normalizes before sending).
        """

        q = (query_vector.numpy() if isinstance(query_vector, torch.Tensor)
             else np.asarray(query_vector, dtype=np.float64))

        assert q.shape == (self.doc_vecs.shape[1],), \
            "query dimensionality must match the server's embed_dim"

        sims = cosine_similarity(q, self.doc_vecs)
        top_ids = top_k_indices(sims, k)

        return [
            {"id": i, "text": self.contexts[i],
             "score": float(sims[i])}
            for i in top_ids
        ]

def evaluate(server, Q, gold, ks=(1, 5, 10), warmup=10):
    """Recall@{1,5,10} and average per-query latency over the query set."""

    # ---- warmup (untimed) ----
    for q in Q[:warmup]:
        server.search(q, max(ks))

    # ---- timed pass ----
    hits = {k: 0 for k in ks}
    latencies = []
    for qi, qvec in enumerate(Q):
        t0 = time.perf_counter()
        results = server.search(qvec, max(ks)) # single round trip
        latencies.append(time.perf_counter() - t0)

        ids = [r["id"] for r in results]
        for k in ks:
            if gold[qi] in ids[:k]:
                hits[k] += 1

    n = len(Q)
    return ({k: hits[k] / n for k in ks},
            sum(latencies) / n,
            latencies)

In [49]:
contexts, queries, gold = load_corpus_and_queries()

# ---- Part (b): Client/Server split
server = RetrieverServer(contexts, embed_dim=256)

# ---- Client: encode queries, ask the server, measure Recall@10 --
Q_full = encode(queries, is_query=True)     # (200, 768)
Q = truncate(Q_full, 256)                   # (200, 256)

recalls, avg_lat, lat = evaluate(server, Q, gold, ks=(1, 5, 10))

D_full = encode(contexts, is_query=False)   # (1000, 768)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Snowflake/snowflake-arctic-embed-m-v1.5
Key                 | Status  | 
--------------------+---------+-
pooler.dense.weight | MISSING | 
pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [50]:
# ---- Part (c): Evaluation
print(f"\nEvaluation over {len(Q)} queries @ 256 dims, "
        f"{NUM_DOCS} docs")
for k in (1, 5, 10):
    print(f"Recall@{k:<2} = {recalls[k]:.3f}")
print(f"Avg query latency (server round trip): "
        f"{avg_lat * 1000:.2f} ms  (median "
        f"{sorted(lat)[len(lat) // 2] * 1000:.2f} ms)")


Evaluation over 200 queries @ 256 dims, 1000 docs
Recall@1  = 0.480
Recall@5  = 0.900
Recall@10 = 0.965
Avg query latency (server round trip): 1.51 ms  (median 1.46 ms)


In [51]:
# ---- Part (a): Matryoshka dimensions --------
print(f"{'dim':>5} {'Recall@10':>10}")
for d in [64, 128, 256, 768]:
    D, Q = truncate(D_full, d), truncate(Q_full, d)
    r = recall_at_k(D, Q, gold, TOP_K)
    print(f"{d:>5} {r:>10.3f}")

  dim  Recall@10
   64      0.925
  128      0.960
  256      0.965
  768      0.970


In [60]:
# 3.3 (d) - Timing the plaintext operations

def print_stage_times(times, n):
    """
    Average time per query of each stage
    """

    total = 0.0
    for stage in TIMING_STAGES:
        if stage in times:
            ms = 1000 * sum(times[stage]) / n
            total += ms
            print(f"  {stage:<24}{ms:>10.2f} ms")
        else:
            print(f"  {stage:<24}{'-':>10}")
    print(f"  {'Total:':<24}{total:>10.2f} ms")

Q_256 = truncate(Q_full, 256)
plaintext_times = {stage: [] for stage in ("Encode question:", "Server similarity:", "Top-k selection:", "Fetch and/or decrypt docs:")}

for qi in range(NUM_QUERIES):
    t0 = time.perf_counter()
    truncate(encode([queries[qi]], is_query=True), 256)
    t1 = time.perf_counter()
    similarity = cosine_similarity(Q_256[qi], server.doc_vecs)
    t2 = time.perf_counter()
    top_ids = top_k_indices(similarity, TOP_K)
    t3 = time.perf_counter()
    docs = [server.contexts[i] for i in top_ids]
    t4 = time.perf_counter()
    for stage, dt in zip(plaintext_times, (t1 - t0, t2 - t1, t3 - t2, t4 - t3)):
        plaintext_times[stage].append(dt)

In [61]:
# 3.3 (d) - Printing the times of the plaintext operations
print(f"Average per query over {NUM_QUERIES} queries:")
print_stage_times(plaintext_times, NUM_QUERIES)

plaintext_storage = (server.doc_vecs.numel() * server.doc_vecs.element_size()
                     + sum(len(c.encode("utf-8")) for c in server.contexts))
print(f"\nServer storage: {plaintext_storage / 1e6:.2f} MB\n"
      f"Vectors: {server.doc_vecs.numel() * server.doc_vecs.element_size() / 1e6:.2f} MB\n"
      f"Texts: {sum(len(c.encode('utf-8')) for c in server.contexts) / 1e6:.2f} MB")

Average per query over 200 queries:
  Encode question:             46.00 ms
  Encrypt query:                   -
  Server similarity:            2.08 ms
  Decrypt scores:                  -
  Top-k selection:              0.08 ms
  Fetch and/or decrypt docs:      0.01 ms
  Total:                       48.16 ms

Server storage: 1.76 MB
Vectors: 1.02 MB
Texts: 0.74 MB


### 3.3

In [62]:
class CKKSRetrieverServer:
    """Class which stores the encrypted texts and encrypted and normalized vectors.
    The client interacts with the server by requesting the encrypted scores through the
    scores function and by requesting the ids of a chosen subset (top-k) of documents through
    get_by_ids."""

    def __init__(self, public_context, encrypted_vectors, encrypted_text):
        self.contexts = ts.context_from(public_context)
        self.doc_vecs = [ts.ckks_vector_from(self.contexts, n) for n in encrypted_vectors]
        self.doc_texts = dict(enumerate(encrypted_text))

    def score(self, blobs):
        Q = [ts.ckks_vector_from(self.contexts, n) for n in blobs]
        scores = Q[0] * self.doc_vecs[0]
        for qj, dj in zip(Q[1:], self.doc_vecs[1:]):
            scores += qj * dj
        return scores.serialize()

    def get_by_ids(self, ids):
        return [self.doc_texts[i] for i in ids]

In [63]:
def encrypt(aead, doc_id, text, padded_len):
    """
    Encrypts text with AES-GCM.
    The AES-GCM functions work with bytes, so conversion is needed from strings to bytes.
    """
    raw = text.encode("utf-8")
    plaintext = len(raw).to_bytes(4, "big") + raw.ljust(padded_len, b"\x00")
    nonce = os.urandom(12)  # Sets nonce to 12 bytes
    return nonce + aead.encrypt(nonce, plaintext, f"doc:{doc_id}".encode())

def decrypt(aead, doc_id, blob):
    """
    Decrypts with AES-GCM.
    """
    plaintext = aead.decrypt(blob[:12], blob[12:], f"doc:{doc_id}".encode())
    return plaintext[4:4 + int.from_bytes(plaintext[:4], "big")].decode("utf-8")

def encrypt_query(context, Q_vector):
    return [ts.ckks_vector(context, [float(qj)] * NUM_DOCS).serialize() for qj in np.asarray(Q_vector, dtype=np.float64)]

def encrypted_search(context, aead, server, Q_vector, k, times=None):
    """
    Returns the ids, text and score according to the encrypted search protocol defined in the assignment description:

    First, the query is encrypted and is then sent to the server.
    The server then sends the corresponding scores, which are then decrypted by the client.
    The client then picks the top-k documents according to the decrypted scores.
    """
    t0 = time.perf_counter()
    encrypted_Q = encrypt_query(context, Q_vector)
    t1 = time.perf_counter()
    score_blob = server.score(encrypted_Q)
    t2 = time.perf_counter()
    scores = np.array(ts.ckks_vector_from(context, score_blob).decrypt())
    t3 = time.perf_counter()
    top_ids = top_k_indices(scores, k)
    t4 = time.perf_counter()
    results = [{"id": i, "text": decrypt(aead, i, b), "score": float(scores[i])}
            for i, b in zip(top_ids, server.get_by_ids(top_ids))]
    t5 = time.perf_counter()

    if times is not None:
        for stage, dt in zip(TIMING_STAGES[1:], (t1-t0, t2-t1, t3-t2, t4-t3, t5-t4)):
            times.setdefault(stage, []).append(dt)

    return results

In [64]:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree = 8192,
    coeff_mod_bit_sizes = [60,40,40,60])
context.global_scale = 2**40
context.generate_galois_keys()
aead = AESGCM(AESGCM.generate_key(256))

D_256_truncated = truncate(D_full, 256).double().numpy()  # Renormalizes D at 256 dimensions
padded_len = max(len(c.encode("utf-8")) for c in contexts)

t0 = time.perf_counter()
encrypted_vectors = [ts.ckks_vector(context, dj.tolist()).serialize() for dj in D_256_truncated.T]
t1 = time.perf_counter()
encrypted_texts = [encrypt(aead, i, c, padded_len) for i, c in enumerate(contexts)]
t2 = time.perf_counter()
public_context = context.serialize(save_secret_key=False)
CKKSServer = CKKSRetrieverServer(
    public_context,
    encrypted_vectors,
    encrypted_texts)

In [68]:
# 3.3 (d) - Measuring one time setup costs and server storage

vector_bytes = sum(len(b) for b in encrypted_vectors)
text_bytes = sum(len(b) for b in encrypted_texts)
print(f"Encrypting {NUM_DOCS} documents: {t2 - t0:.2f} s\n"
      f"Vectors: {t1 - t0:.2f} s\n"
      f"Texts: {t2 - t1:.3f} s\n")
print(f"Server storage: {(vector_bytes + text_bytes + len(public_context)) / 1e6:.2f} MB\n"
      f"Vector ciphertexts: {vector_bytes / 1e6:.2f} MB\n"
      f"Encrypted texts: {text_bytes / 1e6:.2f} MB\n"
      f"Public context: {len(public_context) / 1e6:.2f} MB")

Encrypting 1000 documents: 1.62 s
Vectors: 1.62 s
Texts: 0.004 s

Server storage: 124.22 MB
Vector ciphertexts: 84.83 MB
Encrypted texts: 4.10 MB
Public context: 35.29 MB


In [72]:
# 3.3 (c) - Finds recall@1/5/10

Q_256_truncated = truncate(Q_full, 256).double().numpy()  # Renormalizes Q at 256 dimensions

hits = {k: 0 for k in (1, 5, 10)}
encrypted_times = {"Encode question:": []}
for qi in range(NUM_QUERY_SUBSET):
    t0 = time.perf_counter()
    truncate(encode([queries[qi]], is_query=True), 256)
    encrypted_times["Encode question:"].append(time.perf_counter() - t0)

    ids = [r["id"] for r in encrypted_search(context, aead, CKKSServer, Q_256_truncated[qi], 10, times=encrypted_times)]
    for k in hits:
        hits[k] += gold[qi] in ids[:k]

In [73]:
print(f"\nEvaluation over {NUM_QUERY_SUBSET} queries @ 256 dims, "
        f"{NUM_DOCS} docs, with the CKKS protocol")
for k in hits:
    print(f"Recall@{k:<2} = {hits[k] / NUM_QUERY_SUBSET:.3f}")


Evaluation over 200 queries @ 256 dims, 1000 docs, with the CKKS protocol
Recall@1  = 0.480
Recall@5  = 0.900
Recall@10 = 0.965


In [74]:
# 3.3 (d) - Prints the times for every stage of the CKKS protocol

print(f"\n Average time per query over {NUM_QUERY_SUBSET} queries:")
print_stage_times(encrypted_times, NUM_QUERY_SUBSET)


 Average time per query over 200 queries:
  Encode question:             44.17 ms
  Encrypt query:             1603.53 ms
  Server similarity:          919.79 ms
  Decrypt scores:               1.43 ms
  Top-k selection:              0.05 ms
  Fetch and/or decrypt docs:      0.09 ms
  Total:                     2569.06 ms


In [75]:
# 3.3 (c) - Computes the maximum absolute difference between encrypted and plain-text scores

encrypted_scores = [np.array(ts.ckks_vector_from(context, CKKSServer.score(
                  encrypt_query(context, Q_256_truncated[qi]))).decrypt())
              for qi in range(NUM_QUERY_SUBSET)]
plaintext_scores = [cosine_similarity(Q_256_truncated[qi], server.doc_vecs) for qi in range(NUM_QUERY_SUBSET)]

max_diff = max(np.abs(e - p).max() for e, p in zip(encrypted_scores, plaintext_scores))

In [76]:
print(f"Max absolute difference over {NUM_QUERY_SUBSET} queries x {NUM_DOCS} docs: {max_diff:.2e}")

Max absolute difference over 200 queries x 1000 docs: 2.18e-07


In [77]:
# 3.3 (c) - Compares the top-k results to check if they're the same

same = 0
for qi in range(NUM_QUERY_SUBSET):
    encrypted_top = top_k_indices(encrypted_scores[qi], TOP_K)
    plaintext_top = [r["id"] for r in server.search(Q_256_truncated[qi], TOP_K)]
    if encrypted_top == plaintext_top:
        same += 1
    else:
        print(f"Query {qi} mismatch: encrypted {encrypted_top}\n          plaintext {plaintext_top}")
print(f"Identical top-{TOP_K}s: {same}/{NUM_QUERY_SUBSET} queries")

Identical top-10s: 200/200 queries


In [79]:
# 3.3 (e) - Times single operations

N_RUNS = 100

op_context = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree = 8192,
                        coeff_mod_bit_sizes = [60, 40, 40, 60])
op_context.global_scale = 2**40
op_context.generate_galois_keys()

v1 = D_256_truncated[:, 0]
v2 = D_256_truncated[:, 1]
encrypted_v1 = ts.ckks_vector(op_context, v1)
encrypted_v2 = ts.ckks_vector(op_context, v2)
v2_list = v2.tolist()

evaluator = seal.Evaluator(op_context.seal_context().data)
galois_keys = op_context.galois_keys().data
rotated = seal.Ciphertext()

def avg_time(op):
    t0 = time.perf_counter()
    for i in range(N_RUNS):
        op()
    return (time.perf_counter() - t0) / N_RUNS

operations = {
    "Ciphertext x Ciphertext:": (lambda: encrypted_v1 * encrypted_v2,
                                lambda: v1 * v2),
    "Ciphertext x Plaintext:": (lambda: encrypted_v1 * v2_list,
                                lambda: v1 * v2),
    "Rotation:": (lambda: evaluator.rotate_vector(encrypted_v1.ciphertext()[0], 1, galois_keys, rotated),
                  lambda: np.roll(v1, -1)),
    "Addition:": (lambda: encrypted_v1 + encrypted_v2,
                  lambda: v1 + v2)
}

In [94]:
print(f"Average over {N_RUNS} runs, vectors of {len(v1)} values:")
print(f"  {'Operation:':<26}{'CKKS':>12}{'NumPy':>12}")
for name, (ckks_op, numpy_op) in operations.items():
    t_ckks, t_numpy = avg_time(ckks_op), avg_time(numpy_op)
    print(f"  {name:<26}{t_ckks * 1e3:>9.3f} ms{t_numpy * 1e6:>9.2f} us") # Using microseconds to measure NumPy vector operation times

Average over 100 runs, vectors of 1000 values:
  Operation:                        CKKS       NumPy
  Ciphertext x Ciphertext:      3.168 ms     6.74 us
  Ciphertext x Plaintext:       1.068 ms     3.24 us
  Rotation:                     2.094 ms    90.42 us
  Addition:                     0.467 ms     5.61 us
